In [ ]:
library(ggplot2)
library(Seurat)
library(cowplot)
library(RColorBrewer)
library(getopt)
library(ComplexHeatmap)
library(pheatmap)
library(viridis)
# library(future)
library(dplyr)
library(Matrix)
library(data.table)
library(reshape2)
library(ggpubr)
library(tidyr)
# library(harmony)
library(tidyverse)
library(ggrepel)

cols = c(brewer.pal(9, "Set1"),brewer.pal(8,"Set2")[1:8],brewer.pal(12,"Paired")[1:12],brewer.pal(8,"Dark2")[1:8],brewer.pal(8,"Accent"),brewer.pal(12, "Set3"),brewer.pal(9,"Pastel1"),brewer.pal(8,"Pastel2"))
cols = c(cols,cols)

# Fig.S2e

In [ ]:
seuobj <- readRDS("public_data/Visium.20chips.area_anno.250525.rds")
binall <- readRDS("mydata_spatial.rds")

In [ ]:
pub_deg = read.table("public_data/area.allmarker.txt",sep='\t',header=T)
pub_top = pub_deg %>% group_by(cluster) %>% top_n(100,avg_log2FC)

genelist = read.csv('area.DEG.csv',check.nmes=F)
top_genes = genelist %>% group_by(cluster) %>% top_n(100,avg_log2FC)
GENE = intersect(unique(pub_top$gene),unique(top_genes$gene))

In [ ]:
av1 <-AverageExpression(seuobj,group.by = "area",assays = "RNA")
av2 <-AverageExpression(binall,group.by = "area_m",assays = "RNA")

In [ ]:
x <- av1$RNA[GENE,]
y <- av2$RNA[GENE,]
x <- t(scale(t(x)))  # gene-wise z-score
y <- t(scale(t(y)))
mat <- cor(x, y, method="spearman")

In [ ]:
av1$RNA <- av1$RNA[, c("CA1","CA2","CA3","CA4","DG","FAS","SLRM")]
av2$RNA <- av2$RNA [,c("CA1","CA2","CA3","CA4","DG","FAS","SLRM")]
options(repr.plot.width=9, repr.plot.height=10) 
pdf(paste0("/data/work/09_FigureV3/plot/data_corrlation.pdf"),height=10,width=12)
p = pheatmap(mat,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         border = FALSE,
         scale = "none",
         fontsize = 18,
         color = colorRampPalette(mycolors)(50),
        # color = colorRampPalette(c("#fafafa", "white", "#005cab"))(50),
         border_color = "grey"
        )
dev.off()

# Fig.S2f-g

In [ ]:
binall = readRDS("/data/users/liuyuyang/online/07.ReAnalysis/02.bin/bin100_12chips.merge.xy_adj.250715.rds")
binall <- NormalizeData(binall)

In [ ]:
marker.list <- read.csv("/data/work/01.result/00.bin100.region_deg/12chips.region.allDEG.csv",check.names=F)

df1 <- marker.list %>%
  pivot_wider(names_from = cluster, values_from = avg_log2FC)
  
df1[is.na(df1)] <- 0
df1 = as.data.frame(df1)
rownames(df1) <- df1$gene
mat <- df1[,-1]
mat[1:3,1:3]

ct.names = colnames(mat)
mat$ct.max = ct.names[apply(mat,1,which.max)]
mat$ct.max=factor(mat$ct.max,levels=c("CA1","CA2","CA3","CA4","DG","FAS","SLRM"))
mat=mat[order(mat$ct.max),]

In [ ]:

ct_levels <- levels(mat$ct.max)
# top_marker.list <- marker.list %>% group_by(cluster) %>% top_n(n = 50, wt = avg_log2FC)

genelist <- lapply(ct_levels, function(ct) {
  # unique(marker.list$gene[which(marker.list$cluster == ct)])
    rownames(mat[which(mat$ct.max == ct), ])
})
names(genelist) <- ct_levels

In [ ]:
avg <- AverageExpression(binall,group.by = 'area_m')
avg <- avg$RNA[, c("CA1","CA2","CA3","CA4","DG","FAS","SLRM")]
Z   <- t(scale(t(log1p(avg)))) 

In [ ]:
regions <- c("CA1","CA2","CA3","CA4","DG","FAS","SLRM") 

In [ ]:
val_matrix <- function(geneset) {
  M <- t(sapply(regions, function(r) {
    gg <- intersect(geneset[[r]], rownames(Z))
    if (length(gg) == 0) return(setNames(rep(NA_real_, length(regions)), regions))
    colMeans(Z[gg, regions, drop = FALSE], na.rm = TRUE)
  }))
  rownames(M) <- regions
  M
}
plot_val <- function(geneset, title) {
  M <- val_matrix(geneset)
  p <- pheatmap(M,
           cluster_rows = FALSE, cluster_cols = FALSE, scale = "none",
           color  = colorRampPalette(rev(brewer.pal(11, "RdBu")))(100),
           breaks = seq(-2, 2, length.out = 101),         # 固定 -2..2 色标
           display_numbers = TRUE, number_format = "%.1f",
           main = title,
           labels_row = paste0(regions, " module"),
           labels_col = paste0(regions, " area"))
    return(p)
    }
    
plot_val(genelist,'heatmap')